# 周度净值材料分析

**数据**：`materials/CTA&混合中性_周度净值序列.xlsx`（工作表 `Sheet`）  
**口径**：`docs/consensus.md`（周收益、Pearson、弱相关阈值 |r|<0.30）  
**技能**：pandas 向量化、`loc`、seaborn 作图（`python-data-notebooks`）；产出可另存图到 `results/`。


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook", palette="colorblind")

def project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in (cwd, cwd.parent):
        if (p / "materials").is_dir():
            return p
    raise FileNotFoundError("Could not find materials/; run from repo root or notebooks/")

REPO = project_root()
XLSX = REPO / "materials" / "CTA&混合中性_周度净值序列.xlsx"
RESULTS = REPO / "results"
RESULTS.mkdir(parents=True, exist_ok=True)


In [ ]:
raw = (
    pd.read_excel(XLSX, sheet_name="Sheet")
    .rename(
        columns={
            "净值日期": "as_of",
            "对冲9号累计净值": "nav_hedge9",
            "CTA实盘净值": "nav_cta",
        }
    )
    .assign(as_of=lambda d: pd.to_datetime(d["as_of"]))
    .sort_values("as_of")
    .reset_index(drop=True)
)
raw.head()


## 数据质量检查
查看缺失与类型；CTA 列在早期周次可为空。


In [ ]:
raw.info()
missing = raw.isna().sum()
missing.rename("missing_count").to_frame()


## 周收益率（向量化）
仅在两列净值均存在时保留；对 `nav_hedge9`、`nav_cta` 使用 `pct_change()`。


In [ ]:
both = raw.dropna(subset=["nav_hedge9", "nav_cta"]).copy()
both["ret_hedge9"] = both["nav_hedge9"].pct_change()
both["ret_cta"] = both["nav_cta"].pct_change()
rets = both.dropna(subset=["ret_hedge9", "ret_cta"]).loc[:, ["as_of", "ret_hedge9", "ret_cta"]]
rets.describe().T


## Pearson 相关与散点图
与 `docs/consensus.md` §2.2 一致；弱相关判定 §2.3（|r|<0.30）。


In [ ]:
r_pearson = rets["ret_hedge9"].corr(rets["ret_cta"], method="pearson")
r_pearson


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sns.scatterplot(data=rets, x="ret_hedge9", y="ret_cta", ax=ax, alpha=0.55)
ax.axhline(0, color="gray", lw=0.8)
ax.axvline(0, color="gray", lw=0.8)
ax.set_title(f"周收益散点（Pearson r = {r_pearson:.4f}）")
ax.set_xlabel("对冲9号周收益率")
ax.set_ylabel("CTA 周收益率")
fig.tight_layout()
fig.savefig(RESULTS / "correlation_scatter_seaborn.png", dpi=150)
plt.show()


## 结论
若 |r| < 0.30，按项目共识表述为**弱线性相关**。可将 `rets` 导出到 `results/` 以便复核：
`rets.to_csv(RESULTS / "weekly_returns_pandas.csv", index=False)`


In [ ]:
rets.to_csv(RESULTS / "weekly_returns_pandas.csv", index=False)
print("Saved", RESULTS / "weekly_returns_pandas.csv")
